notes

In [ ]:
"""
    Story
    1. Select by eye positions in quadrants, this looks like that in the position timeline
    2. Estimate receptive fields and calculate significant components with bootsstrapping test. Some neurons look  very fine and similar, but most neurons do not look like that.
    3. Next we fit a optic flow field, to estimate the Focus of Contraction. That looks like
    that.
    4. When we apply this to all neurons, we get a distribution for the horizontal positions.
    These distributions do not look different, allthough they are statistically different.
    So we have confusing argument right now, that from optic inspection there is no shift,
    and also the distribution does not look different. But statistically we get an effect.
    5. Next steps are to try to improve data quality. Since dataset is splittet by eye position there is a lot less data being averaged than it was previously,
    estimations are a lot noisier, but the experiment can be rerun with a older fish, which shows more stable eye movements, a longer recording time to have more data, and developing more strict criteria.
    Also, repeat the experiment in silico to estimate how sensitive this method would be to a potential shift in the receptive field.
"""

In [ ]:
"""
    theoretisch zu tun
1. find translation sensitive neurons
2. calculate delta for all translation sensitive neurons
3. run a statistical test for the delta value
4. create visualizations for deltas and poster
5. show that difference could be shown, if there is one
"""

In [ ]:
"""
6, 7, 8,  43

negative example: 15

in signi plots
89
"""


<ins> Usage of this notebook: </ins> <br>
1. to load data, only variables in the first cell below must be changes, <br>then all cells in 'Load data' can be run to fetch data for one recording.
2. the notebook is structured into four after that: <br> a) running the full pipeline of to the point of editing <br>
b) making some calculations on the output variables of the pipeline before <br> c) all plotting <br>
3. at the end of the notebook is one cell with archived code, this should be kept clean.

select fish & recording and set flags for computations

In [ ]:
plane=0
# 2026-02-25_mb_fish1_rec2
recording_name = '2026-03-04_mb_fish8_rec2'
# recording_name='2026-02-25_mb_fish1_rec2'
imaging_rate=1.9957 # fish8_rec2
# imaging_rate=1.9988 # fish1_rec2
# set flags for saving intermediate results
compute_dff=False # False skip recalculating dff


# load data

In [ ]:


from scipy.optimize import minimize
from scipy.interpolate import interp1d

from preprocess import *
from functions import *
from plot import *

%matplotlib qt # plot in separate windows
matplotlib.style.use('classic') # avoid dark background in plots from PyCharm settings

In [ ]:
recording_path = os.path.join('data', recording_name)
save_path = os.path.join('results', recording_name, 'plane'+str(plane))

load fluorescence traces and stimulus data

In [ ]:
fluorescence, rec, phase, ca_rec_group_id_fun = digest_folder(recording_path, imaging_rate=imaging_rate, plane=plane)

load eyetracking data

In [ ]:
camera = h5py.File(os.path.join(recording_path, 'Camera.hdf5'), 'r')

eye_pos=np.column_stack(
    (np.array(camera['eyepos_ang_le_pos_0']).squeeze(),
     np.array(camera['eyepos_ang_re_pos_0']).squeeze()))

eye_pos_resampled = interp1d(
    np.array(camera['fish_embedded_frame_time']).squeeze(),
    eye_pos.T, kind='nearest')(rec['time_resampled']).T

process recording

In [ ]:
# add resampled CMN data to resampled time domain for each phase
# (info: time domain is resampled from ~2.1Hz to 10Hz in digest_folder())
process_recording(rec, phase, radial_bin_num=16)

either compute the dff or load previously computed dff from disc, resample dff to stimulus time

In [ ]:
if  compute_dff:
    dff_original, dff_resampled = calculate_dff_vectorized(rec, fluorescence, rec['imaging_rate'])
    Path(save_path).mkdir(parents=True, exist_ok=True)
    np.save(os.path.join(save_path, "dff_original.npy"), dff_original)
else:
    dff_original = np.load(os.path.join(save_path, "dff_original.npy"))
    dff_resampled = scipy.interpolate.interp1d(rec['ca_times'], dff_original, kind='nearest')(
            rec['time_resampled'])


# manually define eye position quadrants

In [ ]:
# params=(10,-5, 6, -12) # parameters_fish1_rec2
# params=(8,-8, 7, -9) # parameters_fish8_rec2
params=(.1,.1,-.1,-.1) # parameters_fish8_rec2


In [ ]:
eye_pos=(eye_pos-eye_pos.mean(axis=0))/eye_pos.std(axis=0)

In [ ]:
# 1st quadrant (defined by lower boundaries)
# 3rd quadrant (define by upper boundaries)
q1, q3, out=plot_eyepositions(eye_pos,
                  q1_min_left=params[0],
                  q1_min_right=params[1],
                  q1_width=1,
                  q1_height=1,
                  q3_max_left=params[2],
                  q3_max_right=params[3],
                  q3_widht=1,
                  q3_height=1)



In [ ]:
# Eye tracking~~~~
q1_mask, q3_mask=generate_eyepos_masks(
    eye_pos[:,0],
    eye_pos[:,1],
    camera['fish_embedded_frame_time'],
    rec['time_resampled'],
    q1_min_left = params[0],
    q1_min_right = params[1],
    q3_max_left = params[2],
    q3_max_right = params[3])

In [ ]:
s=2.
time=camera['fish_embedded_frame_time'][:]
time/=60
time-=time[0]
fig, ax = plt.subplots()

ax.scatter(time[q1], eye_pos[:,0][q1], s=s, color='red')
ax.scatter(time[q3], eye_pos[:,0][q3], s=s, color='blue')
ax.scatter(time[out], eye_pos[:,0][out], s=s, color='black')
ax.scatter(time[q1], eye_pos[:,1][q1], s=s, color='red')
ax.scatter(time[q3], eye_pos[:,1][q3], s=s, color='blue')
ax.scatter(time[out], eye_pos[:,1][out], s=s, color='black')
ax.plot(time, np.zeros_like(time), color='grey')

ax.set_xlabel('time (min)', fontsize=20, color='black')
ax.set_ylabel('horizontal eye position (no unit)', fontsize=20, color='black')
ax.tick_params(axis='both', which='major', labelsize=16, color='black')
ax.set_xlabel('time (min)', fontsize=20, color='black')
ax.set_ylabel('horizontal eye position (no unit)', fontsize=20, color='black')
ax.tick_params(axis='both', which='major', labelsize=16, color='black')
fig.set_facecolor('white')
ax.set_facecolor('white')

# fit only to significant parts of receptive fiels

use 2D projections to calculate RSS_angle
if theres time, do this while still n 3D coordinated exclude potential artifacts from the 2D projection

note: scipy minimize, dont care about values, since only angles count.
define rotation axis: azimuth angle, elevation angle



since fitting is only by angle, fits are way too "distracted" by other components without that.

## executions

In [ ]:
radial_bin_centers, positions=rec['radial_bin_centers'], rec['positions']
radial_bin_centers.shape, positions.shape

In [ ]:
# fig=plt.figure()
# ax=fig.add_subplot(projection='3d')
# ax.scatter(rec['positions'][:,0],rec['positions'][:,1],rec['positions'][:,2])
# ax.scatter(0,0,0)
# plt.show()

In [ ]:
# TOF=tof(0, np.pi/4, positions)
# positions=rec['positions']
# TOF.shape, positions.shape

In [ ]:
# # plot in 3D
# ax = plt.figure().add_subplot(projection='3d')
# ax.quiver(positions[:,0], positions[:,1], positions[:,2], TOF[:,0], TOF[:,1], TOF[:,2], length=0.2)
# ax.set_zlabel('elevation[deg]')

In [ ]:
# mock data
# F = project_to_local_2d_vectors(positions, tof(1.,0., positions)[None,:,:]).squeeze()
#np.sum(F*E, axis=1)/ (np.linalg.norm(E, axis=1) * np.linalg.norm(F, axis=1))

In [ ]:
# #E = rec['preferred_vectors'] # RF estimation from ETAs
# E = RF_est_q1

In [ ]:
# mini=minimize(
#     lambda angles: RSSangle_Fto2D(tof(*angles, positions), E, rec['positions']),
#     [0., 0.],
# )
#
# # calculate fitted optic flow field F from optimal parameters
# F_3D=tof(*mini.x, positions)[None,:,:]
# F=project_to_local_2d_vectors(positions, F_3D).squeeze()

In [ ]:
# # Plot estimated RF and fitted TOF/ROF together
# gs_cmn = plt.GridSpec(20, 10)
# fig_cmn = plt.figure(figsize=(20, 10))
# # Plot preferred local motion vectors quiver plot
# ax_quiv = fig_cmn.add_subplot(gs_cmn[:, :])
# # Get position and local motion preference data
# x, y, _ = cart2sph(*positions.T)
#
# Evels=np.linalg.norm(E, axis=1)
# color = matplotlib.colormaps['tab20'](0)
# ax_quiv.quiver(x, y, E[:, 0], E[:, 1],
#                pivot='mid', color=color, width=0.002, scale=Evels.max() * 30)
#
# Fvels=np.linalg.norm(F, axis=1)
# color = matplotlib.colormaps['tab20'](2)
# ax_quiv.quiver(x, y, F[:, 0], F[:, 1],
#                pivot='mid', color=color, width=0.002, scale=Fvels.max() * 30)
#
# ax_quiv.set_xlabel('azimuth [deg]')
# ax_quiv.set_ylabel('elevation [deg]')
# ax_quiv.set_aspect('equal')

In [ ]:
"""
    theoretisch zu tun
1. find translation sensitive neurons
2. calculate delta for all translation sensitive neurons
3. run a statistical test for the delta value
4. create visualizations for deltas and poster
5. show that difference could be shown, if there is one
"""

In [ ]:
"""
    poster
Grafiken
    Mikroskop setup, CMN stimulus -> Even triggered averaging
    Wie sieht ein global motion sensitive RF aus
    Was ist das delta ? Berechnet sich aus Focus of Contraction
    Wie sehen die Ergebnisse aus ?
        Signifikanztest fuer Verteilungen der beiden FoC fuer linke/rechte Augenposition
            oder ? correlation des FoC (bzw) der angle horizontal mit der Augenposition ?
        Histogramm der Verteilung der delta Werte

    """

In [ ]:
"""
    Histogramm der Differenzen der Verteilung der Augenpositionen im Vergleich zu dem Histogramm
    der Differenzen der
"""

# run

In [ ]:
n_neurons=dff_resampled.shape[0]
E_list_q1, E_list_q3 = [], []
F_list_q1, F_list_q3 = [], []
FoC_list_q1, FoC_list_q3 = [], []
FE1_sim_list, FE3_sim_list=[],[]


for i in tqdm(range(n_neurons)):
# for i in tqdm([6, 8]):
    # load bootstrapped etas
    _path=os.path.join(save_path, 'bootstrapped RBEs',)
    q1_rbe_bootstrapped=np.load(os.path.join(_path, f'neuron_{str(i)}_bsRBE_q1.npy'))
    q3_rbe_bootstrapped=np.load(os.path.join(_path, f'neuron_{str(i)}_bsRBE_q3.npy'))

    # calcium events
    rec['signal_selection'], rec['signal_length'], rec['signal_proportion'], rec['signal_dff_mean']\
        =detect_events_with_derivative_generalAPI(
        rec['cmn_phase_selection'],
        dff_resampled[i],
        rec['sample_rate'])

    # ETAs
    # 1. based on data from all eye positions
    radial_bin_norms_q1, radial_bin_etas_q1 = calculate_local_directions_generalAPI(
        rec['cmn_motion_vectors_2d'][rec['signal_selection'] & q1_mask, :, :],
        rec['radial_bin_edges'])
    # 3. based on data from other side/quadrant of eye positions
    radial_bin_norms_q3, radial_bin_etas_q3 = calculate_local_directions_generalAPI(
        rec['cmn_motion_vectors_2d'][rec['signal_selection'] & q3_mask, :, :],
        rec['radial_bin_edges'])

    # bin significances, based on bootstrapped etas
    significances_q1, pvalues_q1 = calculate_directional_significance_generalAPI(
        radial_bin_etas_q1,
        q1_rbe_bootstrapped,
        bernoulli_alpha=.05/(320*16))
    significances_q3, pvalues_q3 = calculate_directional_significance_generalAPI(
        radial_bin_etas_q3,
        q3_rbe_bootstrapped,
        bernoulli_alpha=.05/(320*16))

    # RFs estimates (cluster statistics are not used in this)
    RF_est_q1=calc_preferred_directions_generalAPI(
        radial_bin_etas_q1,
        significances_q1 > 0,
        rec['radial_bin_centers'])
    RF_est_q3=calc_preferred_directions_generalAPI(
        radial_bin_etas_q3,
        significances_q3 > 0,
        rec['radial_bin_centers'])

    # # bin significances for bootstrapped data, based on bootstrapped etas
    # significances_bs_q1, pvalues_bs_q1 = calculate_directional_significance_permutations_generalAPI(
    #     q1_rbe_bootstrapped)
    # significances_bs_q3, pvalues_bs_q3 = calculate_directional_significance_permutations_generalAPI(
    #     q3_rbe_bootstrapped)

    # # find clusters
    # full_indices_q1, unique_indices_q1, bs_cluster_full_indices_q1, bs_cluster_unique_indices_q1=find_clusters_generalAPI(
    #     significances_q1,
    #     significances_bs_q1,
    #     rec['closest_3_position_indices'],)
    # full_indices_q3, unique_indices_q3, bs_cluster_full_indices_q3, bs_cluster_unique_indices_q3=find_clusters_generalAPI(
    # significances_q3,
    # significances_bs_q3,
    # rec['closest_3_position_indices'],)

    # # calculate cluster statistics
    # original_cluster_scores_q1, bs_max_cluster_scores_q1, cluster_significant_indices_q1 = calculate_cluster_significances_generalAPI(
    #     full_indices_q1,
    #     bs_cluster_full_indices_q1,
    #     significances_q1,
    #     significances_bs_q1)
    # # calculate cluster statistics
    # original_cluster_scores_q3, bs_max_cluster_scores_q3, cluster_significant_indices_q3 = calculate_cluster_significances_generalAPI(
    #     full_indices_q3,
    #     bs_cluster_full_indices_q3,
    #     significances_q3,
    #     significances_bs_q3)

    # plot_rf_overview_generalAPI(
    #     radial_bin_etas_q1,
    #     rec['radial_bin_edges'],
    #     significances_q1,
    #     rec['positions'],
    #     rec['patch_corners'],
    #     rec['patch_indices'],
    #     cluster_significant_indices_q1,
    #     RF_est_q1,
    #     unique_indices_q1,
    #     i,save_path=save_path,q=1)
    #
    # plot_rf_overview_generalAPI(
    #     radial_bin_etas_q3,
    #     rec['radial_bin_edges'],
    #     significances_q3,
    #     rec['positions'],
    #     rec['patch_corners'],
    #     rec['patch_indices'],
    #     cluster_significant_indices_q3,
    #     RF_est_q3,
    #     unique_indices_q3,
    #     i,save_path=save_path, q=3)


    # fit E (translational/ rotational optic flow field)
    E1 = RF_est_q1
    mini=minimize(
    lambda angles: RSSangle_Fto2D(tof(*angles, rec['positions']), E1, rec['positions']),
    [0., 0.],
    )
    # calculate fitted optic flow field F from optimal parameters
    F1_3D=tof(*mini.x, rec['positions'])[None,:,:]
    F1=project_to_local_2d_vectors(rec['positions'], F1_3D).squeeze()
    E_list_q1.append(E1)
    F_list_q1.append(F1)
    FoC_list_q1.append(mini.x)

    E3 = RF_est_q3
    mini=minimize(
    lambda angles: RSSangle_Fto2D(tof(*angles, rec['positions']), E3, rec['positions']),
    [0., 0.],
    )
    # calculate fitted optic flow field F from optimal parameters
    F3_3D=tof(*mini.x, rec['positions'])[None,:,:]
    F3=project_to_local_2d_vectors(rec['positions'], F3_3D).squeeze()
    E_list_q3.append(E3)
    F_list_q3.append(F3)
    FoC_list_q3.append(mini.x)

    FE1_sim_list.append(FE_similarity(F1, E1))
    FE3_sim_list.append(FE_similarity(F3, E3))

    # plot_v1(
    #     E1, E3,
    #     F1, F3,
    #     rec['positions'],
    #     alpha_E=.9, alpha_F=.55,
    #     save_path_=save_path,
    #     neuron_num=i)
    # time.sleep(.1)
#
FoC_array_q1=np.concatenate([FoC[:,None].T for FoC in FoC_list_q1])
FoC_array_q3=np.concatenate([FoC[:,None].T for FoC in FoC_list_q3])

In [ ]:
FoC_array_q1[np.abs(FoC_array_q1) > 3.] = np.nan
FoC_array_q3[np.abs(FoC_array_q3) > 3.] = np.nan
# np.isnan(FoC_array_q1).sum(), np.isnan(FoC_array_q3).sum()

In [ ]:
FoC_array_q1 *= 360/(2*np.pi)
FoC_array_q3 *= 360/(2*np.pi)

# code editing

## fit only to significant parts of receptive fiels

use 2D projections to calculate RSS_angle
if theres time, do this while still n 3D coordinated exclude potential artifacts from the 2D projection

note: scipy minimize, dont care about values, since only angles count.
define rotation axis: azimuth angle, elevation angle



since fitting is only by angle, fits are way too "distracted" by other components without that.

In [ ]:
radial_bin_centers, positions=rec['radial_bin_centers'], rec['positions']
radial_bin_centers.shape, positions.shape

In [ ]:
# fig=plt.figure()
# ax=fig.add_subplot(projection='3d')
# ax.scatter(rec['positions'][:,0],rec['positions'][:,1],rec['positions'][:,2])
# ax.scatter(0,0,0)
# plt.show()

In [ ]:
# TOF=tof(0, np.pi/4, positions)
# positions=rec['positions']
# TOF.shape, positions.shape

In [ ]:
# # plot in 3D
# ax = plt.figure().add_subplot(projection='3d')
# ax.quiver(positions[:,0], positions[:,1], positions[:,2], TOF[:,0], TOF[:,1], TOF[:,2], length=0.2)
# ax.set_zlabel('elevation[deg]')

In [ ]:
# mock data
# F = project_to_local_2d_vectors(positions, tof(1.,0., positions)[None,:,:]).squeeze()
#np.sum(F*E, axis=1)/ (np.linalg.norm(E, axis=1) * np.linalg.norm(F, axis=1))

In [ ]:
# #E = rec['preferred_vectors'] # RF estimation from ETAs
# E = RF_est_q1

In [ ]:
# mini=minimize(
#     lambda angles: RSSangle_Fto2D(tof(*angles, positions), E, rec['positions']),
#     [0., 0.],
# )
#
# # calculate fitted optic flow field F from optimal parameters
# F_3D=tof(*mini.x, positions)[None,:,:]
# F=project_to_local_2d_vectors(positions, F_3D).squeeze()

In [ ]:
# # Plot estimated RF and fitted TOF/ROF together
# gs_cmn = plt.GridSpec(20, 10)
# fig_cmn = plt.figure(figsize=(20, 10))
# # Plot preferred local motion vectors quiver plot
# ax_quiv = fig_cmn.add_subplot(gs_cmn[:, :])
# # Get position and local motion preference data
# x, y, _ = cart2sph(*positions.T)
#
# Evels=np.linalg.norm(E, axis=1)
# color = matplotlib.colormaps['tab20'](0)
# ax_quiv.quiver(x, y, E[:, 0], E[:, 1],
#                pivot='mid', color=color, width=0.002, scale=Evels.max() * 30)
#
# Fvels=np.linalg.norm(F, axis=1)
# color = matplotlib.colormaps['tab20'](2)
# ax_quiv.quiver(x, y, F[:, 0], F[:, 1],
#                pivot='mid', color=color, width=0.002, scale=Fvels.max() * 30)
#
# ax_quiv.set_xlabel('azimuth [deg]')
# ax_quiv.set_ylabel('elevation [deg]')
# ax_quiv.set_aspect('equal')

# plotting

## population level histogram

In [ ]:
epr=eye_pos_resampled - eye_pos_resampled.mean(axis=0)

In [ ]:
FoCmean=np.nanmean(np.concatenate((FoC_array_q1, FoC_array_q3), axis=0), axis=0)[0]

In [ ]:
bins1=40
bins2=75

al=.5
fig, (ax1, ax2)=plt.subplots(2, 1, sharex=False)
ax1.hist(epr[q1_mask,0], bins1, color='red')
ax1.hist(epr[q3_mask, 0],bins=bins1, color='blue')
ax2.hist(FoC_array_q1[:,0]-FoCmean, bins=bins2, alpha=al, color='red')
ax2.hist(FoC_array_q3[:,0]-FoCmean, bins=bins2, alpha=al, color='blue')
ax1.set_ylabel('timepoints', fontsize=20, color='black')
ax1.tick_params(axis='both', which='major', labelsize=16, color='black')
ax2.set_xlabel('azimuth [deg]', fontsize=20, color='black')
ax2.set_ylabel('receptive field estimates', fontsize=20, color='black')
ax2.tick_params(axis='both', which='major', labelsize=16, color='black')

In [ ]:
# scipy.stats.wilcoxon(FoC_array_q1[:,0], FoC_array_q3[:,0], nan_policy='omit')

In [ ]:
plot_rf_overview_generalAPI(
    radial_bin_etas_q1,
    rec['radial_bin_edges'],
    significances_q1,
    rec['positions'],
    rec['patch_corners'],
    rec['patch_indices'],
    cluster_significant_indices_q1,
    RF_est_q1,
    unique_indices_q1,
    0,
)

## eye position plots

In [ ]:
ers=eye_pos_resampled[rec['cmn_phase_selection']]
ers=ers-ers.mean(axis=0)
ers.shape

In [ ]:
# ers=(
#     np.lib.stride_tricks.sliding_window_view(
#         ers, window_shape=[2, 1]).mean(axis=(2, 3)))

In [ ]:
ers_df=(np.diff(ers,axis=0, prepend=ers[0,:][None,:]))
ers_df.shape

In [ ]:
plt.hist(ers_df[:,0].flatten(), bins=1000, alpha=.5)
plt.hist(ers_df[:,1].flatten(), bins=1000, alpha=.5)


In [ ]:
plt.plot(ers)
plt.plot(ers_df)
plt.plot(np.zeros_like(ers[:,0]))

In [ ]:
plt.plot(ers)
plt.plot(ers_df)
plt.plot(np.zeros_like(ers[:,0]))

In [ ]:
x=rec['time_resampled'][rec['cmn_phase_selection']]
y=ers-ers.mean(axis=0)
mask=q1_mask[rec['cmn_phase_selection']]
x.shape, y.shape, mask.shape

In [ ]:
start_idx

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, y, label="sin(x)")
plt.plot(x, np.zeros_like(ers[:,0]))

# Find start and end indices of True regionss
# changes = np.diff(mask.astype(int))
# start_idx = np.where(changes == 1)[0] + 1
# end_idx = np.where(changes == -1)[0] + 1

start_idx = np.where(np.logical_or(
    np.abs(ers_df)[:,0]>10,
    np.abs(ers_df)[:,1]>10,
))[0]

# Handle edges
if mask[0]:
    start_idx = np.r_[0, start_idx]
# if mask[-1]:
#     end_idx = np.r_[end_idx, len(mask)]

# Shade regions
for s in start_idx:
    ax.axvspan(x[s], x[s]+7, color="orange", alpha=0.3)
# # Shade regions
# for s, e in zip(start_idx, end_idx):
#     ax.axvspan(x[s], x[e-1], color="orange", alpha=0.3)

ax.legend()
plt.show()

#

***

ARCHIVE

# plot estimated receptive fields

In [ ]:
plot_v1(
    E1, E3,
    F1, F3,
    rec['positions'],
    alpha_E=.9, alpha_F=.55,
    save_path_=save_path,
    neuron_num=i)

## calculate similarity between fitted optic flow field and estimated RF

In [ ]:
def FE_similarity(F, E):
    return ((np.sum(F*E, axis=1)/
            (np.linalg.norm(np.clip(E, 0.0000001, 1.), axis=1) *
             np.linalg.norm(F, axis=1)))
            .mean())

In [ ]:
FE_similarity(F1, E1)

CODE ARCHIVE

# subselect neurons

## 8% threshold for neural activity

In [ ]:
proportions=[]
for dff_i in dff_resampled:
    signal_selection, signal_length, signal_proportion, signal_dff_mean\
        =detect_events_with_derivative_generalAPI(
        rec['cmn_phase_selection'],
        dff_i,
        rec['sample_rate'])
    proportions.append(signal_proportion)

In [ ]:
prop=np.array(proportions)

In [ ]:
plt.hist(prop, bins=80)

In [ ]:
phase_visual = [[name, phase[name]['__visual_name']] for name in phase.keys() if name.startswith('phase')]
phase_visual = np.array(phase_visual)
phase_visual

## based on response intensitites > 3

In [ ]:
phase_id=99

In [ ]:
dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1).shape

In [ ]:
(dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1)-
 dff_original[:,phase['in_phase_idcs_'+str(phase_id)][0]]).shape

In [ ]:
R=np.stack([np.maximum(0,
                       dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1)-
                       dff_original[:,phase['in_phase_idcs_'+str(phase_id)][0]]
                       )
            for phase_id in range(105)], axis=1)
R.shape

In [ ]:
strong_response_indices_mask = R.max(axis=0)>3
strong_response_indices_mask.sum()

In [ ]:
plt.hist(R.flatten(), bins=2000)

## exlude with drift or whitening from first to last 16 seconds

In [ ]:
t = rec['time_resampled']
t0 = t[0]

In [ ]:
# masks for fist and last 16 seconds
start, end= (t-t[0] < 16), (t > t[-1] - 16.)

In [ ]:
mu_end, mu_start=dff_resampled[:,end].mean(axis=1), dff_resampled[:,start].mean(axis=1)
std_end, std_start=dff_resampled[:,end].std(axis=1), dff_resampled[:,start].std(axis=1)

In [ ]:
np.sum(np.abs(mu_start-mu_end)>4*mu_start), sum((std_end/std_start) > 4)

# tutorial for parallel processing


In [ ]:
def compute_eta(a, b):
    return 0

In [ ]:
bootstrap_num = 1000
for i in range(bootstrap_num):
    shuffled_events = ...
    eta = compute_eta(shuffled_events, ...)

In [ ]:
import concurrent.futures

In [ ]:
motion_vectors = np.ones((30000, 2), dtype=np.float32)
all_shuffled_events = ...
with concurrent.futures.ProcessPoolExecutor(max_workers=12) as exe:
    futures = [exe.submit(
        compute_eta,
        all_shuffled_events[i],
        motion_vectors,)
        for i in range(bootstrap_num)]

    etas = np.array([f.result() for f in futures])